# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

### 환각 Hallucination
LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**

1. **학습 데이터 한계**

   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**

   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**

   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**

* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**

   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**

   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**

   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**

   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.


In [1]:
%pip install pypdf tavily-python faiss-cpu sentence-transformers

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   --------------------- ------------------ 10.0/18.9 MB 51.6 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 54.1 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 45.8 MB/s  0:00:00
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB ?  0:00:00

   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------- ----------------------------- 1/4 [faiss-cpu]
   ---------- ----------------------------- 1/4 [faiss-cpu]
   ---------- ----------------------------- 1/4 [faiss-cpu]
   -------------------- ------------------- 2/4 [tavily-python]
   ------------------------------ --------- 3/4 [sentence-transformers]
   ------------------------------ --------- 3/4 [sentence-transformers]
   ------------------------------ --------- 3/4 [sente


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Document

Document는 LangChain 프레임워크에서 다양한 데이터 소스(예: 텍스트 파일, PDF, 웹페이지 등)로부터 불러온 정보를 표준화된 객체로 표현하는 핵심 데이터 구조이다. 이 객체는 언어 모델(LLM)이 외부 데이터를 이해하고 처리할 수 있도록 도와준다.

**Document 객체의 구조**
1. page_content: 문서의 실제 내용을 담고 있는 문자열(str)이다. 예를 들어, 텍스트 파일의 본문이나 PDF의 텍스트 등이 여기에 저장된다.
2. metadata: 문서에 대한 부가 정보를 담는 딕셔너리(dict) 형태의 속성이다. 예를 들어, 파일 경로, 페이지 번호, 작성자, 데이터 출처 등 다양한 메타데이터를 저장할 수 있다.


**Document의 역할과 활용**
1. 표준화된 데이터 구조: 다양한 포맷의 데이터를 일관된 방식으로 표현하여, LLM이 손쉽게 접근하고 활용할 수 있도록 한다.
2. 문서 처리의 기본 단위: LangChain의 문서 로더(Document Loader)는 파일, 웹, 데이터베이스 등 여러 소스에서 데이터를 읽어와 Document 객체로 변환한다.
3. 청크 단위 분할: 대용량 문서는 작은 단위(청크)로 쪼개어 각각의 Document로 저장하고, 검색 및 임베딩 처리에 활용한다.


In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content='이것은 문서의 내용입니다.',
    metadata={
        'source' : 'ABC pdf',
        'page' : 20,
        'author' : '김철수',
        'date' : '2025-07-09'
    }
)

print(doc)
print(doc.page_content)
print(doc.metadata)

page_content='이것은 문서의 내용입니다.' metadata={'source': 'ABC pdf', 'page': 20, 'author': '김철수', 'date': '2025-07-09'}
이것은 문서의 내용입니다.
{'source': 'ABC pdf', 'page': 20, 'author': '김철수', 'date': '2025-07-09'}


## Document Loader

Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |


### WebBaseLoader


In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()
docs

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nArtificial intelligence - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nGoals\n\n\n\n\nToggle Goals subsect

In [4]:
print(len(docs))
doc = docs[0]
print(doc.metadata)
print(doc.page_content)

1
{'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}




Artificial intelligence - Wikipedia


























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance

















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
Goals




Toggle Goals subsection





1.1
Reasoning and problem-solving








1.2
Knowledge representation








1.3
Planning and decision-making








1.4
Learning








1.5
Natural language processing








1.6
Perception








1.7
Social intelligence








1.8
Ge

### PDF Loader

In [5]:
# PyPDFLoader : PDF 파일을 페이지별로 로드하여 각 페이지르르 개별 문서로 처리한다.
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('data/The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()
print(len(docs))

35


In [6]:
doc = docs[2]
print(doc.page_content)
print(doc.metadata)

The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter
{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'data/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}


### TavilySearchAPIRetriever

https://tavily.com

`TavilySearchAPIRetriever`는 사용자가 입력한 검색어를 바탕으로 Tavily 검색 API를 호출하고, 관련성 높은 웹 문서를 검색 결과 형태로 가져온다.

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily_retriver = TavilySearchAPIRetriever(k=3)     # 최대 3건
docs = tavily_retriver.invoke('삼성전자 주가')
print(len(docs))

3


In [12]:
for doc in docs:
    print(doc)

page_content='오늘 삼성전자 주가는 283,000입니다. 삼성전자은(는) 어떤 증권 거래소에서 거래되나요? 삼성전자은' metadata={'title': 'Samsung Electronics Co Ltd 오늘의 주가 | 005930 실시간 티커', 'source': 'https://kr.investing.com/equities/samsung-electronics-co-ltd', 'score': 0.9999398, 'images': []}
page_content='# 270,500 +1.69% | 삼성전자. KRW-BTC 1,050.576. KRW-DOGE 1억 5,503만. ### 삼성전자 파업 전운에 노동부 중재 역할…중노위 사후 조정 타진 SBS뉴스 1시간 전### 삼성전자 경영진 “열린 자세로 협의할 것…미래 경쟁력 손실되지 않게 해달라” 경향신문 2시간 전### “삼성전자 눈부신 성과엔 정부·협력사 협조 있어” 경향신문 2시간 전. +1.69% 4,500 ㆍ 거래량 80,226,640 거래대금 21조 7,341억. 2026.05.07(목)시 273,000 (-0.36%)고 289,000 (+5.47%)저 260,000 (-5.11%)종 270,500 (-1.28%). 단순이동평균 5 10 20 60 120. 거래량 8,022만 6,640(-17.85%). 주요 매출은 스마트폰, 네트워크시스템, 컴퓨터 등을 생산하는 IM부문에서 발생하고 있으며 반도체, CE 부문이 뒤를 잇고 있다. TV, 스마트폰, 반도체 및 디스플레이 패널 부문 등에서 글로벌 우위의 경쟁력을 확보하고 있으며, 메타버스와 로보틱스 시장으로 진출하기 위해 M&A를 계획하고 있다. Image 39: 종목 로고 ### ‘우리’는 없고 ‘끼리’만 있다…노동 전문가들이 본 삼성전자 노조 ‘분배 투쟁’ 경향신문 1시간 전SBS뉴스 ### 노동장관 "삼성전자 위해 수많은 협력업체 노력…노사 대화하길" SBS뉴스 1시간 전. ## 업종 내 비교. |  | 현재 종목 | 업종 평균 | 업종 

In [13]:
# RAG 간단 구현
# 1. 사용자 검색
# 2. tavily search를 이용한 웹 검색 -> prompt의 context 보강
# 3. 보강된 prompt를 통해 llm에 질의 -> 답변 생성
# 4. 환각 현상을 완화한 정확한 응답

from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

tavily_retriver = TavilySearchAPIRetriever(k=5)

prompt = PromptTemplate.from_template('''
사용장의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
Context:
{context}
Queston:
{question}
''')

llm = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

# 로더/리트리버의 검색 결과를 하나의 텍스트로 변환하는 함수
def format_docs(docs:list[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in docs])

# 체인 구성
tavily_chain = tavily_retriver | format_docs
chain = (
    {"question":RunnablePassthrough(), "context":tavily_chain}
    | prompt
    | llm
    | output_parser
)

# 체인 실행
answer = chain.invoke("신대방삼거리역 근처 맛집 알려줘.")
print(answer)

신대방삼거리역 근처 맛집으로 다음과 같은 곳들을 추천드립니다:

1. **카츠디나인 신대방삼거리 본점**  
   - 주소: 서울특별시 동작구 국사봉1길 12-5 1층  
   - 특징: 겉은 바삭하고 속은 촉촉한 카츠가 인기 메뉴입니다.  
   - 주차: 매장 앞 2대 내외 가능  
   - 분위기: 아이와 함께, 데이트, 모임하기 좋은 곳  

2. **모둠회 식당 (황금특선 메뉴 추천)**  
   - 위치: 서울 동작구 상도로 48 1층, 신대방삼거리역 3번 출구 도보 3~4분  
   - 메뉴: 모둠회 황금특선(약 45,000원) - 회, 곰피, 회무침, 멍게 등 여러 해산물과 함께 즐길 수 있음  
   - 영업시간: 평일 오후 오픈, 주말 점심 오픈, 월요일 휴무  

3. **미분당 신대방삼거리점**  
   - 주소: 서울특별시 동작구 보라매로19길 25 1층  
   - 특징: 깔끔한 쌀국수 맛집, 점심시간에 빠른 식사가 가능  
   - 분위기: 조용한 편  

4. **서민준밀밭**  
   - 주소: 서울특별시 동작구 상도동 324-25  
   - 메뉴: 콩국수, 바지락칼국수, 들깨수제비 등  
   - 특성: 조용한 분위기, 아이와 함께, 모임하기 좋은 장소  
   - 영업시간: 매일 11:00~21:00 (일요일 휴무)  

5. **쿠우쿠우 골드 보라매공원점**  
   - 주소: 서울특별시 동작구 신대방동 395-69 보라매아카데미타워 7층  
   - 특징: 초밥 뷔페, 단체회식 및 아이와 함께 방문하기 좋음  
   - 영업시간: 매일 11:00~22:00  

6. **시간을 들이다 (빵집)**  
   - 주소: 서울특별시 동작구 보라매로 80 1층  
   - 메뉴: 크로와상 등 다양한 빵  
   - 특징: 빵 품절 시 조기 종료 가능, 아이와 함께, 데이트 장소로 적합  

주차 정보와 영업시간 등은 매장별로 상이하니 방문 전 확인을 권해드립니다.  
필요한 음식 종류나 분위기, 방문 목적에 따라 선택하시면 좋습

## Embedding Model

### OpenAIEmbeddings

In [15]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

text = "철수는 골든리트리버를 키우고 있다."
vec = embeddings.embed_documents(text)
print(len(vec))

pd.Series(vec,name="embedding")

19


0     [0.007282257080078125, -0.005168914794921875, ...
1     [0.033355712890625, -0.02587890625, 0.01875305...
2     [-0.00992584228515625, 0.0133514404296875, 0.0...
3     [-0.01385498046875, -0.0027008056640625, 0.014...
4     [0.020660400390625, 0.0614013671875, -0.006847...
5     [0.0291595458984375, -0.0031375885009765625, 0...
6     [0.01032257080078125, -0.020843505859375, -0.0...
7     [0.0145721435546875, -0.00553131103515625, 0.0...
8     [0.01032257080078125, -0.020843505859375, -0.0...
9     [0.00882720947265625, 0.037017822265625, 0.005...
10    [-0.0012350082397460938, -0.00315093994140625,...
11    [-0.01385498046875, -0.0027008056640625, 0.014...
12    [0.037200927734375, -0.0146942138671875, -0.05...
13    [0.0311737060546875, -0.01216888427734375, -0....
14    [0.0170745849609375, 0.01275634765625, -0.0276...
15    [-0.01385498046875, -0.0027008056640625, 0.014...
16    [-0.031158447265625, -0.042266845703125, -0.07...
17    [6.031990051269531e-05, -0.002353668212890

### HuggingFaceEmbeddings


sentence-transformers/all-MiniLM-L6-v2 모델은 6개의 트랜스포머 레이어로 구성된 MiniLM 구조를 기반으로 하며, 입력된 문장이나 단락을 384차원의 임베딩 벡터로 변환해 의미적 유사도 계산, 문서 검색, 분류 등 다양한 자연어 처리 작업에 활용할 수 있다.

이 모델은 경량화되어 빠른 속도와 적은 메모리 사용이 특징이며, 한 번에 최대 256 워드피스까지 입력할 수 있고(이보다 긴 경우 truncate처리), 다양한 언어를 지원한다.

임베딩된 벡터는 코사인 유사도 등으로 문장 간 의미적 유사성을 비교하는 데 사용되며, 실제 서비스에서 효율적으로 적용할 수 있다.

| 세부모델명                         | 레이어 수 | 임베딩 차원 | 최대 입력 길이 | 특징                    |
|-----------------------------------|:--------:|:----------:|:-------------:|-------------------------|
| all-MiniLM-L6-v2                  |    6     |    384     |     256       | 경량, 빠른 속도, 다국어 지원 |
| all-MiniLM-L12-v2                 |   12     |    384     |     256       | 더 깊은 레이어, 더 높은 성능  |


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

text = "철수는 골든리트리버를 키우고 있다."
vec = embeddings.embed_query(text)
print(len(vec))

pd.Series(vec,name="embedding")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


0      0.011022
1      0.065050
2      0.048188
3     -0.068704
4      0.014492
         ...   
379    0.067011
380    0.011610
381    0.025551
382   -0.018868
383    0.005621
Name: embedding, Length: 384, dtype: float64

## Vector Store

벡터 데이터베이스란 쉽게 말해, **비정형 데이터(텍스트, 이미지, 오디오 등)를 숫자 벡터로 변환하여 저장하고, 이 벡터들 간의 유사성을 바탕으로 데이터를 검색**하는 데이터베이스를 말한다. 여기서 벡터는 데이터를 다차원 공간에서 표현한 수학적 객체이다.

- **벡터**: 데이터의 특징을 다차원으로 표현한 값.
  - 예: 단어 임베딩은 단어를 벡터로 변환하여 유사한 단어들이 가까이 위치.
- **벡터 데이터베이스 필요성**:
  - RDBMS는 구조화된 데이터(테이블 형태)에 적합.
  - AI/머신러닝의 발전으로 비정형 데이터를 처리할 필요 증가.
  - 벡터 데이터베이스는 **유사도 기반 검색**으로 고차원 데이터 처리에 유리.

**주요 특징:**
- 유사한 데이터를 빠르게 검색.
- AI 응용 분야(이미지 검색, 자연어 처리, 추천 시스템 등)에서 중요.

- **벡터 데이터베이스와 RDBMS의 주요 차이점**

| **특징**                | **RDBMS**                                                                 | **벡터 데이터베이스**                                                                                  |
|-------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------|
| **데이터 구조**          | 테이블 형식으로 데이터 저장, SQL을 사용하여 질의.                             | 다차원 벡터 형식으로 데이터 저장, 벡터 간 유사도 계산 방식 사용.                                        |
| **검색 방식**            | 키-값 쌍이나 고정 조건 기반 검색 (정확한 일치 검색).                          | 유사성 검색 수행, 벡터 간 거리(예: 코사인 유사도, 유클리드 거리)를 기준으로 유사한 데이터를 반환.         |
| **비정형 데이터 처리**   | 텍스트, 숫자 등 구조화된 데이터 처리에 적합.                                 | 이미지, 오디오, 영상 등 비정형 데이터를 벡터로 변환해 처리 가능.                                       |
| **응용 분야**            | 전통적인 CRUD 작업, 금융 데이터, 고객 데이터 관리 등.                       | AI 기반 추천 시스템, 이미지 검색, 자연어 처리, 음성 인식 등.                                           |
| **확장성**               | 수평 확장 가능하지만 고차원 데이터나 복잡한 쿼리 처리에는 한계.               | 수백만~수십억 개 벡터 데이터를 효율적으로 처리 가능.                                                  |

**벡터 데이터베이스의 주요 특징**

1. **Approximate Nearest Neighbor (ANN) 검색**  
   - **ANN 알고리즘**을 사용해 유사한 벡터를 빠르게 검색.  
   - 검색 속도가 빠르고, 대규모 데이터셋에서도 효율적으로 동작.

2. **확장성**  
   - 수백만~수십억 개의 벡터 데이터를 처리할 수 있는 구조로 설계.  
   - 대규모 데이터셋에서 고속 검색 및 처리가 가능.

3. **유연성**  
   - 텍스트, 이미지, 오디오 데이터를 임베딩 형태로 변환해 저장 가능.  
   - 다양한 머신러닝 모델과 통합하여 사용자 요구에 맞는 검색 시스템 구축 가능.

**주요 벡터 데이터베이스 비교**

| 이름 | 특징 | 장점 | 단점 |
|---|---|---|---|
| Chroma | LLM/RAG 애플리케이션 개발에 자주 사용되는 오픈소스 벡터 데이터베이스. 로컬 환경과 Python 노트북에서 간단히 사용하기 좋다. | 설정이 간단하고 LangChain 실습과 프로토타입 제작에 적합하다. | 초대규모 운영 환경에서는 별도의 운영 설계나 관리형 서비스 검토가 필요할 수 있다. |
| Pinecone | 완전 관리형 벡터 데이터베이스 서비스. 대규모 AI 애플리케이션의 운영 환경에 적합하다. | 서버 관리 부담이 적고, 확장성과 운영 편의성이 좋다. | 오픈소스가 아니며, 서비스 사용 비용이 발생한다. |
| Weaviate | 오픈소스 벡터 데이터베이스. 객체 데이터와 벡터를 함께 저장하고, 다양한 모델 및 모듈과 통합할 수 있다. | 벡터 검색, 하이브리드 검색, 모듈 기반 확장에 강점이 있다. | 기능이 많은 만큼 초기 개념과 설정을 익히는 데 시간이 필요할 수 있다. |
| Faiss | Meta에서 개발한 고성능 벡터 유사도 검색 라이브러리. 벡터 데이터베이스라기보다 검색 인덱스 라이브러리에 가깝다. | 빠른 유사도 검색, GPU 지원, 대규모 벡터 검색 실험에 적합하다. | 자체적으로 DB 서버, API, 권한 관리, 메타데이터 관리 기능을 제공하는 형태는 아니므로 별도 구현이 필요하다. |
| Qdrant | Rust 기반 오픈소스 벡터 검색 엔진/벡터 데이터베이스. API 중심으로 사용하며 payload 기반 필터링을 지원한다. | 성능, 필터링, API 기반 운영에 강점이 있다. | Chroma에 비해 초급 실습에서는 설정과 개념이 조금 더 무겁게 느껴질 수 있다. |


**선택 가이드**

1. **수업 실습, 로컬 테스트, 간단한 RAG 프로토타입**: Chroma, Faiss
2. **운영 환경에서 관리 부담을 줄이고 싶은 경우**: Pinecone
3. **오픈소스 기반으로 확장 가능한 벡터 DB를 구성하고 싶은 경우**: Weaviate, Qdrant
4. **고성능 벡터 검색 실험 또는 GPU 기반 검색이 필요한 경우**: Faiss
5. **메타데이터 필터링과 API 기반 검색 서버 구성이 중요한 경우**: Qdrant

### FAISS

- **공식 문서**: [https://faiss.ai/](https://faiss.ai/)
- **Github**: [https://github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)

**Faiss(Vector Search Library)** 는 Meta의 FAIR(Fundamental AI Research) 그룹에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델이 생성한 임베딩 벡터 검색 등에 활용된다.

**주요 특징**
1. **효율적인 유사도 검색**
   - 쿼리 벡터와 가까운 `k`개의 벡터를 찾는 `k-NN(k-Nearest Neighbors)` 검색을 효율적으로 수행한다.
   - 유사도 기준으로는 L2 거리, Inner Product 등을 사용할 수 있다.
   - 코사인 유사도는 보통 벡터를 정규화한 뒤 Inner Product를 사용하는 방식으로 처리할 수 있다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도, 검색 속도, 메모리 사용량 사이의 균형을 조절할 수 있다.
   - 데이터가 커질수록 전체 벡터를 모두 비교하는 방식은 비효율적이므로, 인덱스를 활용해 검색 범위를 줄이고 속도를 높인다.

3. **확장성**
   - 대규모 벡터 집합을 다루기 위한 다양한 인덱스와 압축 기법을 제공한다.
   - GPU 병렬 처리와 압축 기법을 활용하면 매우 큰 규모의 벡터 검색도 효율적으로 처리할 수 있다.

4. **유연성**
   - Python과 C++ API를 제공한다.
   - Python/Numpy 인터페이스를 제공하므로, PyTorch나 Scikit-learn 등에서 만든 임베딩 벡터와 함께 사용하기 쉽다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행한다.
   - 저장된 모든 벡터와 직접 비교하므로 정확도가 높다.
   - 데이터가 많아질수록 검색 속도가 느려질 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터 공간을 여러 클러스터로 나누고, 쿼리와 가까운 일부 클러스터만 탐색하여 검색 속도를 높인다.
   - 전체 벡터를 모두 비교하지 않기 때문에 대규모 데이터 검색에 적합하다.
   - 탐색할 클러스터 수를 조절하여 정확도와 속도 사이의 균형을 맞출 수 있다.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이는 방식이다.
   - 원본 벡터를 그대로 저장하는 것보다 메모리를 적게 사용한다.
   - 대규모 벡터 검색에서 빠른 근사 유사도 검색을 수행할 수 있다.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반의 ANN(Approximate Nearest Neighbor) 인덱스이다.
   - 벡터들을 그래프 구조로 연결하고, 가까운 벡터를 빠르게 찾아간다.
   - 검색 속도가 빠르며, 대규모 유사도 검색에 자주 사용된다.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 임베딩 벡터로 변환한 후, 질문과 가장 관련 있는 문서를 검색한다.
   - RAG 시스템에서 사용자의 질문과 관련된 문서를 찾는 데 활용할 수 있다.

2. **이미지 검색**
   - 이미지에서 추출한 특징 벡터를 사용하여 비슷한 이미지를 검색한다.
   - 예를 들어, 사용자가 업로드한 이미지와 유사한 이미지를 찾는 기능에 사용할 수 있다.

3. **추천 시스템**
   - 사용자 행동, 상품 정보, 콘텐츠 정보를 벡터화하여 유사한 항목을 추천한다.
   - 사용자의 관심사와 가까운 상품, 문서, 영상 등을 찾는 데 활용할 수 있다.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석한다.
   - 비슷한 의미나 특징을 가진 데이터들을 그룹으로 묶을 때 사용할 수 있다.

In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/The_Adventures_of_Tom_Sawyer.pdf")
docs = loader.load()

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [20]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

vector_db.save_local("./db/faiss")

In [21]:
# 저장된 vector db 로드
vector_db = FAISS.load_local(
    './db/faiss',
    embeddings=embeddings,
    allow_dangerous_deserialization=True        # 신뢰할 수 있는 pkl 파일 역직렬화 허용
)

vector_db

In [22]:
# 벡터 스토어 조회
# - similarity_search : 유사도 기반 검색 , 단건 조회 저수준 api
search_results = vector_db.similarity_search(
    query="학교 선생님이 아끼는 해부학 책을 누가 찢었는가?",
    k=4     # 기번값
)
search_results

[Document(id='5fd8c84e-30e2-49ce-ab10-e016d444a57b', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'data/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 15, 'page_label': '16'}, page_content='talking about it. Becky wanted to talk to Tom, but he \ndidn’t look at her. \nThen Tom talked to Amy. Becky watched him and she \nwas angry. She said to her friends, “I’m going to have an \nadventure day. You can come on my adventure.” But she \ndidn’t ask Tom. \nLater in the morning, Tom ta lked to Amy again. Becky \ntalked to her friend Alfred and looked at a picture-book \nwith him. Tom watched them and he was angry with \nBecky. \nIn the afternoon, Tom waited for Becky at the school \nfence. He said, “I’m sorry.” \nBut

In [23]:
for i,doc in enumerate(search_results):
    print(f'{i}번째: page {doc.metadata['page']}')
    print(doc.page_content)
    print('-'*60)

0번째: page 15
talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have an 
adventure day. You can come on my adventure.” But she 
didn’t ask Tom. 
Later in the morning, Tom ta lked to Amy again. Becky 
talked to her friend Alfred and looked at a picture-book 
with him. Tom watched them and he was angry with 
Becky. 
In the afternoon, Tom waited for Becky at the school 
fence. He said, “I’m sorry.” 
But Becky didn’t listen to him. She walked into the 
school room. The teacher’s new book was on his table. 
This book wasn’t for children, but Becky wanted to look 
at it. She opened the book quietly and looked at the 
pictures. 
Suddenly, Tom came into the room. Becky was 
surprised. She closed the book quickly, and it tore. Becky 
was angry with Tom and quickly went out of the room. 
Then the children and the teacher came into the room 
and went to their places. The t

### VectorStoreRetriever

`Retriever`는 사용자의 질의(Query)를 받아 관련 있는 `Document` 목록을 반환하는 LangChain의 검색 인터페이스이다.  
그중 `VectorStoreRetriever`는 Chroma, FAISS 같은 Vector Store의 검색 기능을 Retriever 인터페이스로 감싼 객체이다.

즉, Vector Store를 직접 다루지 않고도 LangChain 체인 안에서 일관된 방식으로 문서를 검색할 수 있게 해준다.  
`VectorStoreRetriever`는 벡터DB 자체가 아니라, Vector Store의 검색 기능을 LangChain에서 재사용하기 쉽게 만들어주는 어댑터(Adapter)에 가깝다.


**VectorStoreRetriever의 역할**

1. **관련 문서 검색**
   - 사용자의 질문과 관련성이 높은 문서를 Vector Store에서 검색한다.
   - 일반적으로 질문도 임베딩 벡터로 변환한 뒤, 저장된 문서 벡터와의 유사도를 비교한다.
   - 검색 결과는 LangChain의 `Document` 객체 목록으로 반환된다.

2. **검색 기능의 표준화**
   - Chroma, FAISS, Qdrant 등 Vector Store마다 내부 동작 방식은 다를 수 있다.
   - 하지만 Retriever로 변환하면 LangChain 체인에서는 동일한 방식으로 사용할 수 있다.
   - 덕분에 Vector Store를 바꾸더라도 전체 체인 구조를 크게 변경하지 않아도 된다.

3. **RAG 체인과의 연결**
   - RAG에서는 LLM이 답변을 생성하기 전에 질문과 관련된 문서를 먼저 검색한다.
   - `VectorStoreRetriever`는 이때 필요한 문서를 찾아 Prompt의 Context로 전달하는 역할을 한다.
   - LLM은 검색된 문서를 참고하여 답변을 생성한다.


**VectorStoreRetriever의 작동 흐름**

1. **문서 준비**
   - 웹 문서, PDF, 텍스트 파일 등을 로드한다.
   - 필요한 경우 긴 문서를 여러 조각으로 분할한다.

2. **임베딩 생성**
   - 분할된 문서를 임베딩 모델을 사용해 벡터로 변환한다.
   - 이 단계에서 문서의 의미가 숫자 벡터로 표현된다.

3. **Vector Store 저장**
   - 문서 벡터와 원본 문서 정보를 Vector Store에 저장한다.
   - Chroma, FAISS 같은 도구를 사용할 수 있다.

4. **Retriever 변환**
   - 준비된 Vector Store를 Retriever 형태로 변환한다.
   - 이후 LangChain 체인에서 검색 단계로 사용할 수 있다.

5. **질의 검색**
   - 사용자의 질문이 들어오면 질문도 벡터로 변환된다.
   - 질문 벡터와 문서 벡터 간의 유사도를 계산하여 관련 문서를 찾는다.

6. **LLM 답변 생성에 활용**
   - 검색된 문서는 Prompt의 Context로 전달된다.
   - LLM은 해당 문서를 참고하여 답변을 생성한다.


**Retriever의 주요 유형**

1. **Sparse Retriever**
   - 전통적인 키워드 기반 검색 방식이다.
   - 단어의 등장 빈도나 통계적 중요도를 기준으로 문서를 검색한다.
   - 대표적인 방식으로 TF-IDF, BM25 등이 있다.
   - 정확한 키워드가 포함된 문서를 찾는 데 유리하다.
   - 단, 표현이 다르면 의미가 비슷해도 찾기 어렵다.

2. **Dense Retriever**
   - 문장이나 문서를 임베딩 벡터로 변환한 뒤, 벡터 간 유사도를 기준으로 검색하는 방식이다.
   - OpenAI Embeddings, sentence-transformers, BERT 계열 모델 등을 사용할 수 있다.
   - 단어가 정확히 일치하지 않아도 의미가 비슷하면 관련 문서로 검색될 수 있다.
   - `VectorStoreRetriever`는 일반적으로 Dense Retriever 방식에 가깝다.


**VectorStoreRetriever의 검색 방식**

1. **Similarity Search**
   - 질문 벡터와 가장 유사한 문서를 검색하는 기본 방식이다.
   - 유사도 점수가 높은 문서를 상위 결과로 반환한다.

2. **MMR(Maximal Marginal Relevance) Search**
   - 관련성이 높은 문서를 가져오면서도 결과 간의 중복을 줄이는 방식이다.
   - 비슷한 내용의 문서만 반복해서 검색되는 상황을 줄이는 데 도움이 된다.

3. **Similarity Score Threshold**
   - 일정 유사도 점수 이상인 문서만 반환하는 방식이다.
   - 관련성이 낮은 문서를 제외하고 싶을 때 사용할 수 있다.


**VectorStoreRetriever의 활용 예시**

1. **RAG 기반 질의응답**
   - 사용자의 질문과 관련 있는 문서를 먼저 검색한다.
   - 검색된 문서를 LLM에게 Context로 제공하여 답변의 근거로 사용한다.

2. **문서 기반 챗봇**
   - PDF, 웹 페이지, 사내 문서 등을 Vector Store에 저장한 뒤 질문에 맞는 문서를 검색한다.
   - LLM은 검색된 문서를 바탕으로 답변한다.

3. **학습 자료 검색**
   - 강의 자료, 매뉴얼, 노트 등을 벡터화하여 저장한다.
   - 사용자가 질문하면 관련 개념이 포함된 자료를 찾아준다.

4. **의미 기반 검색**
   - 사용자의 질문과 문서의 단어가 정확히 일치하지 않아도 의미가 비슷한 문서를 찾을 수 있다.
   - 예를 들어 “요금 환불 기준”이라고 질문해도 “취소 정책”, “환불 규정”이 포함된 문서를 검색할 수 있다.

In [24]:
retriever = vector_db.as_retriever(
    search_type = 'similarity',
    search_kwargs = {
        'k':3
    }
)

retriever_result = retriever.invoke('마을 무덤의 남자를 누가 죽였는가?')
retriever_result

[Document(id='6019cbbc-26bd-4feb-b8cd-625035d96b13', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'data/The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 22, 'page_label': '23'}, page_content='Two hundred men looked for Tom and Becky in the \ncave. They looked for three days, but they didn’t find \nthem. People in the town were very sad. \nChapter 9    Huck’s Adventure \n \nHuck didn’t go on Becky’s adventure. He stayed home \nand watched Injun Joe’s house that night. At eleven \no’clock Injun Joe and his friend came out and walked \ndown the street. There was a box in his friend’s hands. \nHuck said quietly, “Maybe that’s the treasure box.” He \nwent after the two men. \nThey walked to Mrs. Douglas’s house and 

In [25]:
for i,doc in enumerate(retriever_result):
    print(f'{i}번째: page {doc.metadata['page']}')
    print(doc.page_content)
    print('-'*60)

0번째: page 22
Two hundred men looked for Tom and Becky in the 
cave. They looked for three days, but they didn’t find 
them. People in the town were very sad. 
Chapter 9    Huck’s Adventure 
 
Huck didn’t go on Becky’s adventure. He stayed home 
and watched Injun Joe’s house that night. At eleven 
o’clock Injun Joe and his friend came out and walked 
down the street. There was a box in his friend’s hands. 
Huck said quietly, “Maybe that’s the treasure box.” He 
went after the two men. 
They walked to Mrs. Douglas’s house and stopped in her 
yard. Huck stayed behind some small trees. The men 
talked, and Huck listened to them. 
Injun Joe was angry. “I want to kill her,” he said to his 
friend. “Mr. Douglas was bad to me. He’s dead now, but I 
remember.” 
“’There are a lot of lights in the house. Maybe her 
friends are visiting,” Injun Joe’s friend said. “We can 
come back tomorrow.” 
“No,” Injun Joe said. “Let’s wait now.” 
Huck liked Mrs. Douglas because she was always good 
to him. He 

### Retrieval Chain

In [27]:
prompt = PromptTemplate.from_template('''
사용자의 질문에 제공된 문서 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
Context:
{context}

Question:
{question}
''')

llm = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

chain = (
    {'context':retriever|format_docs,'question':RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

result = chain.invoke("마을 무덤의 남자를 누가 죽였는가?")
result

'마을 무덤의 남자는 Injun Joe가 죽였습니다. Tom이 증언하길, Injun Joe가 의사를 칼로 찔러 죽였고, Muff Potter는 범인이 아니었습니다.'

## 실습 문제: 음식 리뷰 데이터 기반 RAG 체인 만들기

제공된 `amazon_food_reviews_1k.csv` 파일을 사용하여 음식 리뷰 기반 RAG 체인을 완성하시오.

### 실습 목표

음식 리뷰 데이터를 벡터 DB에 저장하고, 사용자의 질문과 관련된 리뷰를 검색한 뒤, 검색 결과를 바탕으로 LLM이 답변하도록 구성한다.

### 구현할 기능

1. 리뷰 데이터를 Document로 변환하시오.
   - 리뷰 제목, 평점, 리뷰 본문을 검색 대상 텍스트로 구성한다.
   - 리뷰 ID, 상품 ID, 평점, 작성일은 metadata로 구성한다.

2. Document를 FAISS 벡터 DB로 저장하시오.
   - 음식 리뷰 Document 목록을 임베딩한다.
   - FAISS 벡터 DB를 생성한다.
   - 생성한 벡터 DB를 로컬에 저장한다.

3. retriever를 만들고 검색 결과를 확인하시오.
   - FAISS 벡터 DB를 retriever로 변환한다.
   - 사용자의 질문과 관련된 리뷰가 검색되는지 확인한다.
   - 검색 결과의 metadata와 page_content를 함께 출력한다.

4. retriever + prompt + llm + output_parser를 연결하여 RAG 체인을 완성하시오.
   - retriever가 검색한 리뷰를 context로 전달한다.
   - 사용자의 질문을 question으로 전달한다.
   - LLM이 음식 리뷰 context를 기반으로 답변하도록 구성한다.


### 검색 결과 확인 문제

아래 질문을 사용하여 retriever 검색 결과를 먼저 확인하시오.

1. Find reviews saying the taste is good, delicious, or satisfying.
   - 맛이 좋고 만족스럽다는 리뷰를 찾아줘.

2. Find reviews mentioning shipping problems, packing problems, or damaged items.
   - 배송이나 포장에 문제가 있었다는 리뷰를 찾아줘.

3. Find low-rated reviews saying disappointed, stale, bad taste, or waste of money.
   - 평점이 낮고 실망했다는 리뷰를 찾아줘.


### RAG 체인 확인 문제

아래 질문을 사용하여 최종 RAG 체인의 답변을 확인하시오.

1. How do positive reviews describe the taste of the food?
   - 맛에 대한 긍정적인 평가는 어떤 식으로 나타나는가?

2. What complaints are mentioned about shipping or packaging?
   - 배송이나 포장에 대한 불만은 어떤 내용이 있는가?

3. What are the main reasons for dissatisfaction in low-rated reviews?
   - 평점이 낮은 리뷰들은 주로 어떤 이유로 불만을 말하고 있는가?

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS

loader = CSVLoader("data/amazon_food_reviews_1k.csv",encoding='utf-8-sig')
docs = loader.load()

In [27]:
from langchain_core.documents import Document

documents = []

for doc in docs:
    text = doc.page_content

    row_data = {}

    for line in text.split("\n"):
        key, value = line.split(":", 1)
        row_data[key.strip()] = value.strip()

    page_content = f"""
리뷰 제목 : {row_data['Summary']}
평점 : {row_data['Score']}
리뷰 본문 : {row_data['Text']}
"""
    metadata = {
        "review_id": row_data["Id"],
        "product_id": row_data["ProductId"],
        "score": row_data["Score"],
        "date": row_data["Time"]
    }

    review_doc = Document(
        page_content=page_content,
        metadata=metadata
    )

    documents.append(review_doc)

In [28]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vector_db = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)
vector_db.save_local("./db/amazon_reviews")

In [30]:
vector_db = FAISS.load_local(
    './db/amazon_reviews',
    embeddings=embeddings,
    allow_dangerous_deserialization=True        # 신뢰할 수 있는 pkl 파일 역직렬화 허용
)

In [31]:
retriever = vector_db.as_retriever(
    search_type = 'similarity',
    search_kwargs = {
        'k':3
    }
)

In [33]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

In [34]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template('''
사용자의 질문에 제공된 문서 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
Context:
{context}

Question:
{question}
''')

llm = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

chain = (
    {'context':retriever|format_docs,'question':RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

result = chain.invoke("맛이 좋고 만족스럽다는 리뷰를 찾아줘.")
result

"맛이 좋고 만족스럽다는 리뷰는 다음과 같습니다:\n\n1. 리뷰 제목: pleased  \n   평점: 5  \n   리뷰 본문: very tasteful smells great and taste great not very strong but thats how i like it\n\n2. 리뷰 제목: These are AWESOME.  \n   평점: 5  \n   리뷰 본문: Insanely delicious. Texture and taste are phenomenal, they are incredibly satisfying (and don't leave you feeling mildly nauseous like actual junk food) and the ingredients are just marvelous. Mmmm mmm.\n\n3. 리뷰 제목: Ridiculously delicious  \n   평점: 5  \n   리뷰 본문: Ah, yes, Chicken in a Biskit. That's right, folks. Chicken. IN. A. BISKIT. Need I even say more? Look, I grew up on these babies. Every single bite filled me with enough joy to fill the world's oceans three times over. In short, scrumptious, appetizing, tantalizing, succulent, mouthwatering, delectable, delightful, palatable, divine, pleasurable, gratifying, enjoyable, and quite frankly, ridiculously delicious. Fact."

### 정답

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# CSV 파일 읽기
df = pd.read_csv("data/amazon_food_reviews_1k.csv")

In [ ]:
# 리뷰 데이터를 Document 객체로 변환
documents = []

for _, row in df.iterrows():
    page_content = f"""
리뷰 제목: {row["Summary"]}
평점: {row["Score"]}
리뷰 본문: {row["Text"]}
""".strip()

    metadata = {
        "review_id": row["Id"],
        "product_id": row["ProductId"],
        "score": row["Score"],
        "time": row["Time"],
    }

    documents.append(
        Document(
            page_content=page_content,
            metadata=metadata
        )
    )

In [ ]:
# Embedding 모델 준비
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# FAISS 벡터 DB 생성
vector_db = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

# 생성한 FAISS 벡터 DB를 로컬에 저장
vector_db.save_local("./db/faiss_food_reviews")

# 저장된 FAISS 벡터 DB 다시 불러오기
vector_db = FAISS.load_local(
    "./db/faiss_food_reviews",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
# retriever 생성
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

# 검색 결과를 보기 좋게 출력하는 함수
def print_retrieval_results(question, korean):
    results = retriever.invoke(question)

    print("=" * 80)
    print("질문:", question)
    print("해석:", korean)
    print("=" * 80)

    for i, doc in enumerate(results, start=1):
        print(f"\n[{i}] 검색 결과")
        print("metadata:", doc.metadata)
        print("page_content:")
        print(doc.page_content)
        print("-" * 80)

# retriever 검색 결과 확인
retrieval_questions = [
    (
        "Find reviews saying the taste is good, delicious, or satisfying.",
        "맛이 좋고 만족스럽다는 리뷰를 찾아줘."
    ),
    (
        "Find reviews mentioning shipping problems, packing problems, or damaged items.",
        "배송이나 포장에 문제가 있었다는 리뷰를 찾아줘."
    ),
    (
        "Find low-rated reviews saying disappointed, stale, bad taste, or waste of money.",
        "평점이 낮고 실망했다는 리뷰를 찾아줘."
    ),
]

for question, korean in retrieval_questions:
    print_retrieval_results(question, korean)

In [ ]:
# 검색된 Document 목록을 문자열 context로 변환하는 함수
def format_docs(docs):
    return "\n\n".join(
        [
            f"""
[리뷰 {i}]
metadata: {doc.metadata}
내용:
{doc.page_content}
""".strip()
            for i, doc in enumerate(docs, start=1)
        ]
    )

In [ ]:
# RAG용 PromptTemplate 작성
prompt = PromptTemplate.from_template("""
당신은 음식 리뷰 데이터를 분석하는 AI assistant입니다.

아래 Context는 사용자의 질문과 관련된 음식 리뷰 검색 결과입니다.
반드시 Context에 근거해서 한국어로 답변하세요.
Context에 없는 내용은 추측하지 말고, 알 수 없다고 답변하세요.

Context:
{context}

Question:
{question}

Answer:
""")


# LLM과 OutputParser 준비
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

output_parser = StrOutputParser()

In [ ]:
# retriever + prompt + llm + output_parser를 LCEL 방식으로 연결
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | output_parser
)


# RAG 체인 답변 확인
rag_questions = [
    (
        "How do positive reviews describe the taste of the food?",
        "맛에 대한 긍정적인 평가는 어떤 식으로 나타나는가?"
    ),
    (
        "What complaints are mentioned about shipping or packaging?",
        "배송이나 포장에 대한 불만은 어떤 내용이 있는가?"
    ),
    (
        "What are the main reasons for dissatisfaction in low-rated reviews?",
        "평점이 낮은 리뷰들은 주로 어떤 이유로 불만을 말하고 있는가?"
    ),
]

for question, korean in rag_questions:
    print("=" * 80)
    print("질문:", question)
    print("해석:", korean)
    print("=" * 80)

    answer = rag_chain.invoke(question)

    print(answer)
    print()